# Simple Linear Regression — Google Colab

**Goal:** Predict **GPA** (Grade Point Average) from **SAT** (test score).

| Phase | Topic | Cells |
|-------|-------|-------|
| Phase 0 | Setup | Setup & Imports |
| Phase 1 | Data Pre-processing | Load → Cleaning → Encoding → Split |
| Phase 2 | Algorithm | Train → Predict → Visualize → Evaluate |

> **Run:** Runtime → Run all (or Ctrl+F9)

## Phase 0 — Cell 0: Install Libraries

Google Colab usually includes most libraries. This cell ensures required packages are available.

**What this cell does:** Installs scikit-learn, pandas, matplotlib, numpy, and seaborn quietly.

In [ ]:
!pip install -q scikit-learn pandas matplotlib numpy seaborn

## Phase 0 — Cell 1: Import Libraries

Import libraries for data handling, preprocessing, modeling, and evaluation.

**What this cell does:** Loads numpy, pandas, matplotlib, seaborn, and sklearn modules.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

plt.rcParams['figure.figsize'] = (10, 6)
sns.set_theme(style='whitegrid')
np.random.seed(42)

print('Libraries ready')

---
# Phase 1: Data Pre-processing

Prepare the data before training the model — same template is reused for other algorithms.

## Phase 1 — Cell 1: Load and Explore Data

Load the CSV file and perform initial exploration (head, info, describe, shape).

**What this cell does:** Reads `Datasets/Simple linear regression.csv` and displays basic statistics.

In [ ]:
# Step 1) Load dataset
dataset = pd.read_csv('Datasets/Simple linear regression.csv')

print('First 5 rows:')
display(dataset.head())

print('\nDataset info:')
dataset.info()

print('\nStatistical summary:')
display(dataset.describe())

print(f'\nShape: {dataset.shape[0]} rows x {dataset.shape[1]} columns')

## Phase 1 — Cell 2: Data Cleaning (Handling Missing Values)

Data cleaning includes checking missing values, removing duplicates, and using SimpleImputer when needed.

**What this cell does:** Checks for nulls, drops duplicate rows, and prepares an imputer template for other datasets.

In [ ]:
# Step 2) Data cleaning

# 1) Check missing values
print('Missing values per column:')
print(dataset.isnull().sum())

# 2) Remove duplicate rows
rows_before = len(dataset)
dataset = dataset.drop_duplicates().reset_index(drop=True)
rows_after = len(dataset)
print(f'\nDuplicates removed: {rows_before - rows_after}')

# 3) SimpleImputer — reusable template for other datasets
imputer = SimpleImputer(missing_values=np.nan, strategy='mean')
if dataset.isnull().sum().sum() > 0:
    dataset.iloc[:, :] = imputer.fit_transform(dataset)
    print('Missing values imputed with mean')
else:
    print('No missing values — imputer not applied')

print(f'\nRows after cleaning: {rows_after}')

## Phase 1 — Cell 3: Categorical Data Encoding

ML algorithms require numeric input. Categorical columns are encoded with One-Hot or Label Encoding. For SAT/GPA, all columns are numeric, so encoding is skipped.

**What this cell does:** Detects categorical columns and applies OneHotEncoder if needed; otherwise skips.

In [ ]:
# Step 3) Categorical encoding

cat_cols = dataset.select_dtypes(include=['object', 'category']).columns.tolist()
num_cols = dataset.select_dtypes(exclude=['object', 'category']).columns.tolist()

if cat_cols:
    print(f'Categorical columns found: {cat_cols}')
    preprocessor = ColumnTransformer(
        transformers=[('encoder', OneHotEncoder(handle_unknown='ignore'), cat_cols)],
        remainder='passthrough'
    )
    dataset = pd.DataFrame(
        preprocessor.fit_transform(dataset),
        columns=preprocessor.get_feature_names_out()
    )
    print('One-Hot Encoding applied')
else:
    print('No categorical columns — encoding skipped.')
    print(f'Numeric columns: {num_cols}')

## Phase 1 — Cell 4: Splitting the Data

Define feature X (SAT) and target y (GPA), then split into 80% training and 20% test sets.

**What this cell does:** Creates X, y arrays and applies train_test_split with random_state=42.

In [ ]:
# Step 4) Train-Test split

# X = feature (SAT)
X = dataset[['SAT']].values

# y = target (GPA)
y = dataset['GPA'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'X_train shape: {X_train.shape}')
print(f'X_test shape:  {X_test.shape}')
print(f'y_train shape: {y_train.shape}')
print(f'y_test shape:  {y_test.shape}')

> **Note:** Simple Linear Regression does **not** require feature scaling with a single feature. Scaling is needed for SVR and KNN.

---
# Phase 2: Simple Linear Regression

Apply the simple linear regression algorithm: `GPA = b0 + b1 x SAT`

## Phase 2 — Cell 5: Train the Model

Train `LinearRegression` on the training data. The model learns the slope (b1) and intercept (b0).

**What this cell does:** Fits the regressor and prints intercept and slope coefficients.

In [ ]:
# Step 5) Train model
regressor = LinearRegression()
regressor.fit(X_train, y_train)

print(f'Intercept (b0): {regressor.intercept_:.4f}')
print(f'Slope (b1):       {regressor.coef_[0]:.6f}')
print(f'\nEquation: GPA = {regressor.intercept_:.4f} + {regressor.coef_[0]:.6f} x SAT')

## Phase 2 — Cell 6: Predict

After training, the model predicts GPA values for the training and test sets.

**What this cell does:** Generates y_pred_train and y_pred_test using the fitted model.

In [ ]:
# Step 6) Predict
y_pred_train = regressor.predict(X_train)
y_pred_test = regressor.predict(X_test)

print('Sample predictions (Test set):')
for i in range(min(5, len(y_test))):
    print(f'  SAT={X_test[i][0]:.0f} -> Actual GPA={y_test[i]:.2f}, Predicted={y_pred_test[i]:.2f}')

## Phase 2 — Cell 7: Visualization

Scatter plot of the data with the regression line — training (blue), test (green), line (red).

**What this cell does:** Plots SAT vs GPA with the fitted regression line.

In [ ]:
# Step 7) Visualization
plt.figure(figsize=(10, 6))
plt.scatter(X_train, y_train, color='blue', label='Training set', alpha=0.7)
plt.scatter(X_test, y_test, color='green', label='Test set', alpha=0.7)

# Regression line over full SAT range
X_line = np.linspace(dataset['SAT'].min(), dataset['SAT'].max(), 100).reshape(-1, 1)
y_line = regressor.predict(X_line)
plt.plot(X_line, y_line, color='red', linewidth=2, label='Regression line')

plt.xlabel('SAT Score')
plt.ylabel('GPA')
plt.title('Simple Linear Regression — SAT vs GPA')
plt.legend()
plt.tight_layout()
plt.show()

## Phase 2 — Cell 8: Evaluation

Evaluate the model on the test set using MAE (Mean Absolute Error), RMSE (Root Mean Squared Error), and R² (Coefficient of Determination).

**What this cell does:** Computes and displays regression evaluation metrics.

In [ ]:
# Step 8) Evaluation
mae = mean_absolute_error(y_test, y_pred_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))
r2 = r2_score(y_test, y_pred_test)

results = pd.DataFrame({
    'Metric': ['MAE', 'RMSE', 'R²'],
    'Value': [mae, rmse, r2],
    'Description': [
        'Mean Absolute Error',
        'Root Mean Squared Error',
        'Coefficient of Determination (1 = perfect)'
    ]
})

display(results.round(4))